# ETC-level analysis with etc_summary and climate indices

### Analysis ideas:
- Regress outbreak occurrence vs. climate state
- Frequency modulation (outbreak cyclone counts per teleconnection phase)
- Regress etc_summary structural predictors (min msl, deepening rate, meridional-elongation proxy) on indices to see of ENSO/MJO/SPV directly influence cyclone structure
- Compositing by index state (El Nino OC composites vs La Nina OC composites)

In [1]:
### Importing necessary packages
import pandas as pd
import os
from pathlib import Path
import re
import warnings

#warnings.filterwarnings('ignore')
import matplotlib 
import matplotlib.pyplot as plt
import matplotlib.colors as col
from matplotlib.gridspec import GridSpec
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import cmocean
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import numpy as np
import seaborn as sns
import xarray as xr


## ENSO

#### Oni index obtained from NOAA PSL

In [39]:
etc_summary = pd.read_csv('data/etc_summary_final.csv').drop(columns='Unnamed: 0')
oni = pd.read_csv('data/oni_main.csv', skipfooter=8, engine='python')
oni.columns = ['time', 'ONI']

In [40]:
oni.tail(10)

,time,ONI
906,2025-07-01,-0.14
907,2025-08-01,-0.28
908,2025-09-01,-0.40
909,2025-10-01,-0.51
910,2025-11-01,-0.55
911,2025-12-01,-0.54
912,2026-01-01,-0.37
913,2026-02-01,-0.14
914,2026-03-01,0.13
915,2026-04-01,0.48


In [41]:
oni.columns

Index(['time', 'ONI'], dtype='object')

In [42]:
etc_summary['time_min_msl'] = pd.to_datetime(etc_summary['time_min_msl'])
oni['time'] = pd.to_datetime(oni['time'])

In [43]:
etc_summary['oni_month'] = etc_summary['time_min_msl'].dt.tz_localize(None).dt.to_period('M')
oni['oni_month'] = oni['time'].dt.to_period('M')

In [44]:
etc_summary = etc_summary.merge(
    oni[['oni_month','ONI']],
    on='oni_month',
    how='left',
).drop(columns='oni_month')

In [45]:
etc_summary

,track_id,track_start,track_end,time_min_msl,min_msl,n_steps,n_reports,n_tornado,n_hail,n_wind,n_cores_total,n_cores_kept,kept_reports,max_core_reports,ONI
0,1.0,1995-12-01 00:00:00+00:00,1995-12-06 23:00:00+00:00,1995-12-02 22:00:00+00:00,980.3069,144,0,0,0,0,0,0,0,0,-0.98
1,2.0,1995-12-01 00:00:00+00:00,1995-12-12 03:00:00+00:00,1995-12-10 06:00:00+00:00,961.9612,251,0,0,0,0,0,0,0,0,-0.98
2,3.0,1995-12-03 06:00:00+00:00,1995-12-07 01:00:00+00:00,1995-12-05 04:00:00+00:00,994.3956,92,1,1,0,0,0,0,0,0,-0.98
3,4.0,1995-12-04 04:00:00+00:00,1995-12-06 04:00:00+00:00,1995-12-06 04:00:00+00:00,978.0338,49,0,0,0,0,0,0,0,0,-0.98
4,5.0,1995-12-06 17:00:00+00:00,1995-12-08 01:00:00+00:00,1995-12-07 19:00:00+00:00,980.9462,31,0,0,0,0,0,0,0,0,-0.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2220,6114.0,2025-03-24 18:00:00+00:00,2025-03-29 11:00:00+00:00,2025-03-27 22:00:00+00:00,1005.1210,106,5,0,4,1,1,0,0,0,-0.06
2221,6116.0,2025-03-26 17:00:00+00:00,2025-03-30 19:00:00+00:00,2025-03-29 10:00:00+00:00,982.2738,99,0,0,0,0,0,0,0,0,-0.06
2222,6117.0,2025-03-26 22:00:00+00:00,2025-03-31 23:00:00+00:00,2025-03-31 20:00:00+00:00,992.6212,114,927,66,277,584,7,2,891,821,-0.06
2223,6119.0,2025-03-28 20:00:00+00:00,2025-03-31 21:00:00+00:00,2025-03-30 23:00:00+00:00,989.9344,73,1,0,1,0,0,0,0,0,-0.06


## MJO:

#### Obtained from link in Tippet (2018) paper:
http://www.bom.gov.au/climate/mjo/graphics/rmm.74toRealtime.txt) 

In [4]:
mjo = pd.read_csv('data/rmm.74toRealtime.txt', 
                  sep=r"\s+", 
                  skiprows=2,
                  names=["year", "month", "day","RMM1","RMM2","phase","amplitude", "method"],
                  )

In [7]:
mjo['time'] = pd.to_datetime(mjo[['year','month','day']])

In [8]:
mjo.columns

Index(['year', 'month', 'day', 'RMM1', 'RMM2', 'phase', 'amplitude', 'method',
       'time'],
      dtype='object')

In [52]:
etc_summary_sorted = etc_summary.sort_values('time_min_msl')
mjo_sorted = mjo.sort_values('time')

In [54]:
mjo_sorted

,year,month,day,hour,PC1,PC2,PC1+PC2_amplitude,time
0,1991,1,1,0,0.12526,-0.06945,0.14323,1991-01-01
1,1991,1,2,0,0.18542,-0.04887,0.19175,1991-01-02
2,1991,1,3,0,0.23960,-0.03933,0.24281,1991-01-03
3,1991,1,4,0,0.27446,0.01990,0.27518,1991-01-04
4,1991,1,5,0,0.27714,0.07555,0.28725,1991-01-05
...,...,...,...,...,...,...,...,...
12957,2026,6,23,0,-0.12132,-0.41000,0.42757,2026-06-23
12958,2026,6,24,0,-0.15453,-0.34350,0.37666,2026-06-24
12959,2026,6,25,0,-0.19861,-0.25575,0.32381,2026-06-25
12960,2026,6,26,0,-0.19577,-0.17493,0.26254,2026-06-26


In [ ]:
etc_summary = pd.merge_asof(
    etc_summary_sorted,
    mjo_sorted[['time', 'PC1', 'PC2', 'PC1+PC2_amplitude']],
    left_on='time_min_msl',
    right_on='time',
    direction='nearest'
).drop(columns='time').sort_index()

KeyError: "['RMM1', 'RMM2', 'phase', 'amplitude'] not in index"

## SPV